# QM 640 Capstone — Step 3: Manual Screening & Announcement-Type Classification

This step is deliberately NOT automated end-to-end — the Synopsis's Data
Quality Risk section requires manual review of 8-K text and newswire
language to (a) apply the exclusion criteria and (b) classify
`announcement_type`. This notebook builds the worksheet and, later, checks
inter-rater reliability. **The actual screening happens outside this
notebook, in a spreadsheet.**

## >>> STOP HERE — manual step, outside this notebook <<<

1. Download `screening_worksheet.csv` (or edit it directly on GitHub / in
   Google Sheets after downloading) and manually review each row against
   the exclusion criteria in the Synopsis, using the `filing_url` link to
   read the actual 8-K text.
2. Fill in `is_genuine_ai_event`, `announcement_type`,
   `confounding_event_flag`, `trading_halt_flag`, `sufficient_history_flag`.
3. Re-upload the completed CSV back into `data/raw/screening_worksheet.csv`
   in the repo (via `git push` from your local machine, GitHub's web upload,
   or by re-running Cell 1 in a fresh Colab session, replacing the file, and
   running the push cell below).
4. Give `screening_recode_sample.csv` to an independent reviewer, **without
   your own classifications visible**, and have them fill in
   `recoder_announcement_type` / `recoder_is_genuine_ai_event`. Merge that
   back into the repo the same way.

Once both files are updated in the repo, continue to Part B below.

## Part B — Cohen's kappa on the re-coded sample

Run this after the manual screening and independent re-coding are both
complete and pushed to the repo. Re-run Cell 1 first if this is a new
session, to pull the latest screened data.

In [ ]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"
REPO_NAME = "QM640-WALSH-CAPSTONE"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(f"Clone failed: no .git folder found at {BASE_DIR}")

print("Repo ready at:", BASE_DIR)
!ls {BASE_DIR}/data/raw

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 978, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 978 (delta 71), reused 110 (delta 41), pack-reused 813 (from 1)
Receiving objects: 100% (978/978), 7.54 MiB | 9.72 MiB/s, done.
Resolving deltas: 100% (512/512), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE
cik_ticker_map.csv	    screening_recode_sample.csv  sector_reference.csv
edgar_candidate_events.csv  screening_TO_REVIEW.csv	 sp500_constituents.csv
firm_size.csv		    screening_worksheet.csv


In [ ]:
import pandas as pd
import csv
from sklearn.metrics import cohen_kappa_score
import os
from google.colab import userdata # Added for BASE_DIR dependency

# --- Start of BASE_DIR and related definitions (usually from Cell 419ed1b5) ---
# These definitions are included here to ensure the cell runs independently,
# in case the setup cell (Cell 419ed1b5) was not executed or the kernel was reset.
GITHUB_USERNAME = "Shanmuganathan75"
REPO_NAME = "QM640-WALSH-CAPSTONE"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
BASE_DIR = f"/content/{REPO_NAME}"
# --- End of BASE_DIR and related definitions ---

# Definitions moved from cell 14ac5461 to ensure they are in scope.
RAW_DIR = os.path.join(BASE_DIR, "data/raw")
SCREENING_FILE = os.path.join(RAW_DIR, "screening_worksheet.csv")
RECODE_FILE = os.path.join(RAW_DIR, "screening_recode_sample.csv")

# --- Start of build_worksheet function (moved from Cell fea5bf9f) ---
def build_worksheet():
    src = os.path.join(RAW_DIR, "edgar_candidate_events.csv")
    # Check if edgar_candidate_events.csv exists before proceeding
    if not os.path.exists(src):
        print(f"Error: Required source file '{src}' not found. Please ensure it exists.")
        return pd.DataFrame() # Return empty DataFrame to prevent further errors

    df = pd.read_csv(src)

    df["file_date"] = pd.to_datetime(df["file_date"])
    df = df.sort_values("file_date").drop_duplicates(subset=["cik", "file_date"], keep="first")

    df["is_genuine_ai_event"] = ""       # Y/N - excludes incidental AI mentions
    df["announcement_type"] = ""          # partnership / R&D / M&A
    df["confounding_event_flag"] = ""     # Y/N - other material event in [-2,+2]
    df["trading_halt_flag"] = ""          # Y/N
    df["sufficient_history_flag"] = ""    # Y/N - >=120 trading days pre-event
    df["exclude_reason"] = ""             # free text if excluded
    df["filing_url"] = df.apply(
        lambda r: f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={r['cik']}",
        axis=1,
    )
    df["screener_notes"] = ""

    df.to_csv(SCREENING_FILE, index=False, quoting=csv.QUOTE_ALL)
    print(f"Worksheet built: {len(df)} candidate events -> {SCREENING_FILE}")

    n_subsample = max(1, int(len(df) * 0.20))
    subsample = df.sample(n=n_subsample, random_state=42)
    subsample_out = subsample[["accession_no", "company_name", "file_date"]].copy()
    subsample_out["recoder_announcement_type"] = ""
    subsample_out["recoder_is_genuine_ai_event"] = ""
    subsample_out.to_csv(RECODE_FILE, index=False, quoting=csv.QUOTE_ALL)
    print(f"20% re-coding sample ({n_subsample} events) -> {RECODE_FILE}")
    return df
# --- End of build_worksheet function ---


def calculate_kappa():
    # Fail loud instead of silently rebuilding (and wiping) the worksheet -
    # this silent-rebuild fallback is exactly what caused the earlier data loss.
    if not os.path.exists(SCREENING_FILE) or not os.path.exists(RECODE_FILE):
        raise RuntimeError(
            f"{SCREENING_FILE} or {RECODE_FILE} not found. Run Part A first if this "
            f"is genuinely a fresh start - do NOT auto-rebuild here, since that would "
            f"silently wipe an existing worksheet if the path was just momentarily wrong."
        )

    original = pd.read_csv(SCREENING_FILE)
    recode = pd.read_csv(RECODE_FILE)

    # Fail loud instead of silently computing kappa on duplicated/corrupted data -
    # this is exactly the bug that produced an inflated count earlier.
    assert original["accession_no"].is_unique, (
        f"DUPLICATE accession_no in {SCREENING_FILE} "
        f"({len(original)} rows, {original['accession_no'].nunique()} unique). "
        f"Run 03h_health_check.ipynb before computing kappa."
    )
    assert recode["accession_no"].is_unique, (
        f"DUPLICATE accession_no in {RECODE_FILE} "
        f"({len(recode)} rows, {recode['accession_no'].nunique()} unique). "
        f"Run 03h_health_check.ipynb before computing kappa."
    )

    merged = recode.merge(
        original[["accession_no", "announcement_type"]], on="accession_no", how="left"
    )
    merged = merged.dropna(subset=["announcement_type", "recoder_announcement_type"])
    merged = merged[merged["recoder_announcement_type"] != ""]

    if len(merged) < 2:
        print("Not enough re-coded rows yet. Fill in recoder_announcement_type first.")
        return None

    kappa = cohen_kappa_score(merged["announcement_type"], merged["recoder_announcement_type"])
    print(f"Cohen's kappa (announcement_type, n={len(merged)}): {kappa:.3f}")

    if kappa < 0.70:
        print("\nKAPPA BELOW .70 - per the Synopsis, this triggers a joint review "
              "of the classification criteria before finalizing the full sample.")
        disagreements = merged[merged["announcement_type"] != merged["recoder_announcement_type"]]
        print(disagreements[["company_name", "announcement_type", "recoder_announcement_type"]])
    else:
        print("Kappa meets the .70 threshold - classification is reliable enough to proceed.")
    return kappa


kappa_result = calculate_kappa()

Cohen's kappa (announcement_type, n=97): 1.000
Kappa meets the .70 threshold - classification is reliable enough to proceed.
